# Kaggle Preprocessing Notebook

**Amac:** 2000 ham videoyu uctan uca isleyip egitime hazir `(30, 69)` sekans dosyalarina donusturmek.

### Kullanim:
1. Kaggle > Create > New Notebook > Accelerator: **GPU T4 x2**
2. Add Data > `real life violence situations dataset`
3. Hucreleri sirasiyla calistirin
4. Output sekmesinden `.npy` dosyalarini indirin

### Hucre Haritasi:
| # | Ne Yapar |
|---|---|
| 1 | preprocessing.py modulunu Kaggle ortamina yazar |
| 2 | Kutuphane import + sabitler |
| 3 | Dataset klasorlerini kesfeder |
| 4 | Stratified split (70/15/15) |
| 5 | Tum videolari isler > per-video .npy |
| 6 | Rastgele 5 .npy dogrulama |
| 7 | Sliding window + motion filter > sekanslar |
| 8 | Final dogrulama + ozet |

In [ ]:
# HUCRE 1: preprocessing.py modulunu Kaggle ortamina yaz
# Bu hucre tum preprocessing fonksiyonlarini /kaggle/working/preprocessing.py olarak yazar.
# Sonraki hucrelerde bu modulden import yapilir.

!pip install ultralytics -q

MODULE_CODE = '''
"""
Shared preprocessing functions used by both Kaggle Notebook (offline)
and real-time inference (online).

Pipeline order (decision_log.md D2):
    Conditional CLAHE → GaussianBlur 3×3 → Resize 640×640 → BGR→RGB
    → YOLOv8n-Pose → Multi-Person B+ → Hip Centering → Shoulder-Hip Scaling
    → 69-dim Feature Vector
"""

import os
import math
import cv2
import numpy as np

# ── Constants ────────────────────────────────────────────────
TARGET_FPS = 10
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)
CLAHE_BRIGHTNESS_THRESHOLD = 50
GAUSSIAN_KERNEL_SIZE = (3, 3)
GAUSSIAN_SIGMA = 0
RESIZE_DIM = (640, 640)
NUM_KEYPOINTS = 17
KEYPOINT_CONFIDENCE_THRESHOLD = 0.5
FEATURE_DIM = 69
SKELETON_DIM = 34
TORSO_HEIGHT_EPSILON = 1e-6
SEQUENCE_LENGTH = 30
SLIDING_WINDOW_STRIDE = 15
MOTION_FILTER_THETA = 0.05


# ── Frame Preprocessing (D2) ─────────────────────────────────

def check_low_brightness(frame, threshold=CLAHE_BRIGHTNESS_THRESHOLD):
    """Frame'in ortalama parlaklığı threshold altındaysa True döner."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return gray.mean() < threshold


def apply_clahe(frame):
    """LAB renk uzayında L kanalına CLAHE uygular."""
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l_ch, a_ch, b_ch = cv2.split(lab)
    clahe = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE
    )
    l_ch = clahe.apply(l_ch)
    lab = cv2.merge([l_ch, a_ch, b_ch])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def preprocess_frame(frame):
    """
    Locked preprocessing chain:
    CLAHE (conditional) → GaussianBlur 3×3 → Resize 640×640 → BGR→RGB
    """
    if check_low_brightness(frame):
        frame = apply_clahe(frame)
    frame = cv2.GaussianBlur(frame, GAUSSIAN_KERNEL_SIZE, GAUSSIAN_SIGMA)
    frame = cv2.resize(frame, RESIZE_DIM)
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return frame


# ── Pose Extraction (D3) ─────────────────────────────────────

def extract_pose(frame_rgb, model):
    """
    YOLOv8n-Pose ile frame'den keypoint çıkarır.

    Returns:
        List[dict]: her kişi için {keypoints(17,3), bbox, bbox_area, bbox_center, detection_conf}
    """
    results = model(frame_rgb, verbose=False)
    persons = []

    if results[0].keypoints is not None and len(results[0].keypoints) > 0:
        kps_data = results[0].keypoints.data.cpu().numpy()
        boxes_data = results[0].boxes.data.cpu().numpy()

        for i in range(len(kps_data)):
            kps = kps_data[i]
            box = boxes_data[i]
            bbox_area = (box[2] - box[0]) * (box[3] - box[1])
            cx = (box[0] + box[2]) / 2
            cy = (box[1] + box[3]) / 2
            persons.append({
                'keypoints': kps,
                'bbox': box[:4],
                'bbox_area': bbox_area,
                'bbox_center': (cx, cy),
                'detection_conf': box[4],
            })

    return persons


# ── Multi-Person Selection (D4/D5) ───────────────────────────

def select_top2_persons(persons):
    """Top-2 kişi seçimi (bbox alanına göre) + X-axis sort."""
    if len(persons) == 0:
        return None, None
    if len(persons) == 1:
        return persons[0], None
    sorted_by_area = sorted(persons, key=lambda p: p['bbox_area'], reverse=True)
    top2 = sorted_by_area[:2]
    top2_sorted = sorted(top2, key=lambda p: p['bbox_center'][0])
    return top2_sorted[0], top2_sorted[1]


# ── Keypoint Filtering (D12) ─────────────────────────────────

def filter_keypoints(keypoints):
    """Confidence < 0.5 olan keypoint'leri sıfırlar."""
    filtered = np.zeros((NUM_KEYPOINTS, 2), dtype=np.float32)
    for i in range(NUM_KEYPOINTS):
        if keypoints[i, 2] >= KEYPOINT_CONFIDENCE_THRESHOLD:
            filtered[i, 0] = keypoints[i, 0]
            filtered[i, 1] = keypoints[i, 1]
    return filtered


# ── Skeleton Normalization (D14) ──────────────────────────────

def normalize_skeleton(kps_2d):
    """Hip centering + shoulder-hip scaling."""
    hip_mx = (kps_2d[11, 0] + kps_2d[12, 0]) / 2
    hip_my = (kps_2d[11, 1] + kps_2d[12, 1]) / 2

    for i in range(NUM_KEYPOINTS):
        if kps_2d[i, 0] == 0.0 and kps_2d[i, 1] == 0.0:
            kps_2d[i, 0] = hip_mx
            kps_2d[i, 1] = hip_my

    centered = kps_2d.copy()
    centered[:, 0] -= hip_mx
    centered[:, 1] -= hip_my

    sh_my = (kps_2d[5, 1] + kps_2d[6, 1]) / 2
    torso_h = abs(hip_my - sh_my)

    if torso_h > TORSO_HEIGHT_EPSILON:
        centered /= torso_h
    else:
        return np.zeros((NUM_KEYPOINTS, 2), dtype=np.float32)

    return centered


# ── Feature Vector Builder (D13) ─────────────────────────────

def build_feature_vector(person1, person2, frame_width):
    """69-dim vektör: [skeleton1(34)] + [skeleton2(34)] + [norm_distance(1)]."""
    if person1 is not None:
        sk1 = normalize_skeleton(filter_keypoints(person1['keypoints'])).flatten()
    else:
        sk1 = np.zeros(SKELETON_DIM, dtype=np.float32)

    if person2 is not None:
        sk2 = normalize_skeleton(filter_keypoints(person2['keypoints'])).flatten()
    else:
        sk2 = np.zeros(SKELETON_DIM, dtype=np.float32)

    if person1 is not None and person2 is not None:
        d = math.dist(person1['bbox_center'], person2['bbox_center'])
        nd = d / frame_width if frame_width > 0 else 1.0
    elif person1 is not None:
        nd = 1.0
    else:
        nd = 0.0

    return np.concatenate([sk1, sk2, np.array([nd], dtype=np.float32)])


# ── Video Processing ──────────────────────────────────────────

def process_single_video(video_path, model):
    """
    Tek videoyu uçtan uca işler: FPS sampling → preprocess → pose → feature.

    Returns:
        (np.array shape (n_frames, 69), None) on success
        (None, error_string) on failure
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None, "Video açılamadı"

    native_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if native_fps <= 0 or total_frames <= 0:
        cap.release()
        return None, f"Geçersiz FPS={native_fps} veya frame_count={total_frames}"

    frame_interval = max(1, round(native_fps / TARGET_FPS))
    feature_vectors = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            preprocessed = preprocess_frame(frame)
            persons = extract_pose(preprocessed, model)
            p1, p2 = select_top2_persons(persons)
            fv = build_feature_vector(p1, p2, RESIZE_DIM[0])
            feature_vectors.append(fv)

        frame_idx += 1

    cap.release()

    if len(feature_vectors) == 0:
        return None, "Hiç frame işlenemedi"

    return np.array(feature_vectors, dtype=np.float32), None


# ── Sequence Generation ───────────────────────────────────────

def create_sliding_windows(features, window_size=SEQUENCE_LENGTH, stride=SLIDING_WINDOW_STRIDE):
    """(n_frames, 69) → list of (30, 69) windows."""
    n_frames = features.shape[0]
    windows = []

    if n_frames < window_size:
        padded = np.zeros((window_size, FEATURE_DIM), dtype=np.float32)
        padded[:n_frames] = features
        windows.append(padded)
    else:
        for start in range(0, n_frames - window_size + 1, stride):
            windows.append(features[start:start + window_size])

    return windows


def compute_motion_score(window):
    """Penceredeki ardışık frame çiftleri arası ortalama L2 mesafesi."""
    diffs = np.diff(window, axis=0)
    return np.linalg.norm(diffs, axis=1).mean()


def apply_motion_filter(windows, theta=MOTION_FILTER_THETA):
    """Düşük hareketli Violence pencerelerini eler (D11)."""
    return [w for w in windows if compute_motion_score(w) >= theta]

'''

with open('/kaggle/working/preprocessing.py', 'w') as f:
    f.write(MODULE_CODE)

print("preprocessing.py yazildi -> /kaggle/working/preprocessing.py")

In [ ]:
# HUCRE 2: Import ve Sabitler
# preprocessing.py modulunden fonksiyonlari import eder.
# Notebook'a ozel sabitler (path, split oranlari) burada tanimlanir.

import os
import random
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from ultralytics import YOLO
from sklearn.model_selection import train_test_split

import sys
sys.path.insert(0, '/kaggle/working')
from preprocessing import (
    process_single_video, create_sliding_windows,
    apply_motion_filter, FEATURE_DIM, SEQUENCE_LENGTH
)

# Notebook-only constants
POSE_MODEL_NAME = "yolov8n-pose.pt"
KAGGLE_INPUT = "/kaggle/input/real-life-violence-situations-dataset"
OUTPUT = "/kaggle/working"
VIOLENCE_LABEL = 1
NONVIOLENCE_LABEL = 0
SPLIT_RANDOM_STATE = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

# YOLO model yukle
pose_model = YOLO(POSE_MODEL_NAME)

print("Import tamamlandi. YOLO modeli yuklendi.")

In [ ]:
# HUCRE 3: Dataset Kesfi
# Kaggle input dizininde Violence ve NonViolence klasorlerini bulur.
# Video sayilarini dogrular (beklenen: 1000+1000=2000).

for root, dirs, files in os.walk(KAGGLE_INPUT):
    if "Violence" in dirs or "NonViolence" in dirs:
        VIOLENCE_DIR = os.path.join(root, "Violence")
        NONVIOLENCE_DIR = os.path.join(root, "NonViolence")
        break

violence_videos = sorted(glob(os.path.join(VIOLENCE_DIR, "*")))
nonviolence_videos = sorted(glob(os.path.join(NONVIOLENCE_DIR, "*")))

print(f"Violence:    {len(violence_videos)} video  ->  {VIOLENCE_DIR}")
print(f"NonViolence: {len(nonviolence_videos)} video  ->  {NONVIOLENCE_DIR}")
print(f"Toplam:      {len(violence_videos) + len(nonviolence_videos)}")

In [ ]:
# HUCRE 4: Stratified Split (70 / 15 / 15)
# Videolari train/val/test'e boler (sinif orani korunur).
# split_map dict'i olusturur: {video_path: (split_name, label)}

all_videos = violence_videos + nonviolence_videos
all_labels = [VIOLENCE_LABEL]*len(violence_videos) + [NONVIOLENCE_LABEL]*len(nonviolence_videos)

train_val_f, test_f, train_val_l, test_l = train_test_split(
    all_videos, all_labels, test_size=TEST_RATIO,
    stratify=all_labels, random_state=SPLIT_RANDOM_STATE)

val_adj = VAL_RATIO / (1 - TEST_RATIO)
train_f, val_f, train_l, val_l = train_test_split(
    train_val_f, train_val_l, test_size=val_adj,
    stratify=train_val_l, random_state=SPLIT_RANDOM_STATE)

split_map = {}
for f, l in zip(train_f, train_l): split_map[f] = ('train', l)
for f, l in zip(val_f, val_l):     split_map[f] = ('val', l)
for f, l in zip(test_f, test_l):   split_map[f] = ('test', l)

for name, files, labels in [('train',train_f,train_l),('val',val_f,val_l),('test',test_f,test_l)]:
    v = sum(1 for l in labels if l==1)
    nv = len(labels) - v
    print(f"{name:5s}: {len(files):4d} video  (V={v}, NV={nv})")
    pd.DataFrame({'filepath':files,'label':labels,'split':name}).to_csv(
        os.path.join(OUTPUT, f"{name}.csv"), index=False)

In [ ]:
# HUCRE 5: Tum Videolari Isle > Per-Video .npy
# Her videoyu process_single_video() ile isler:
#   FPS sampling (10fps) > preprocess > YOLOv8n-Pose > multi-person
#   > normalize > 69-dim feature vector > .npy kaydet

for split in ['train', 'val', 'test']:
    for label in ['violence', 'nonviolence']:
        os.makedirs(os.path.join(OUTPUT, 'features', split, label), exist_ok=True)

process_log = []
skipped_videos = []

for video_path in tqdm(all_videos, desc="Video Isleme"):
    split_name, label = split_map[video_path]
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    label_str = 'violence' if label == VIOLENCE_LABEL else 'nonviolence'
    npy_path = os.path.join(OUTPUT, 'features', split_name, label_str, f"{video_name}.npy")

    features, error = process_single_video(video_path, pose_model)

    if features is None:
        skipped_videos.append({'video': video_path, 'error': error})
        continue

    np.save(npy_path, features)
    process_log.append({'video': video_name, 'split': split_name,
                        'label': label_str, 'frames': features.shape[0]})

print(f"\nIslenen: {len(process_log)} | Atlanan: {len(skipped_videos)}")
if skipped_videos:
    for s in skipped_videos[:5]:
        print(f"  SKIP: {s['video']}: {s['error']}")

In [ ]:
# HUCRE 6: Dogrulama - Rastgele 5 .npy Kontrol
# Uretilen .npy dosyalarinin shape, dtype, NaN/Inf ve deger araligini kontrol eder.

npy_files = glob(os.path.join(OUTPUT, 'features', '**', '*.npy'), recursive=True)
print(f"Toplam .npy: {len(npy_files)}")

for f in random.sample(npy_files, min(5, len(npy_files))):
    arr = np.load(f)
    print(f"  {os.path.basename(f):40s} shape={str(arr.shape):12s} "
          f"min={arr.min():.2f}  max={arr.max():.2f}  "
          f"NaN={np.isnan(arr).any()}  Inf={np.isinf(arr).any()}")

In [ ]:
# HUCRE 7: Sliding Window + Motion Filter > Sekanslar
# Her per-video .npy'den 30-frame pencereler olusturur.
# Violence: motion filter (theta=0.05), NonViolence: filtresiz.
# Train: undersampling ile sinif dengesi saglanir.

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUTPUT, 'sequences', split), exist_ok=True)

    v_seqs, nv_seqs = [], []

    for f in glob(os.path.join(OUTPUT, 'features', split, 'violence', '*.npy')):
        windows = create_sliding_windows(np.load(f))
        v_seqs.extend(apply_motion_filter(windows))

    for f in glob(os.path.join(OUTPUT, 'features', split, 'nonviolence', '*.npy')):
        nv_seqs.extend(create_sliding_windows(np.load(f)))

    if split == 'train' and len(nv_seqs) > len(v_seqs):
        random.seed(42)
        nv_seqs = random.sample(nv_seqs, len(v_seqs))

    X = np.array(v_seqs + nv_seqs, dtype=np.float32)
    y = np.array([1]*len(v_seqs) + [0]*len(nv_seqs), dtype=np.float32)

    np.save(os.path.join(OUTPUT, 'sequences', split, f'X_{split}.npy'), X)
    np.save(os.path.join(OUTPUT, 'sequences', split, f'y_{split}.npy'), y)

    print(f"{split:5s}: V={len(v_seqs):5d}  NV={len(nv_seqs):5d}  "
          f"Total={len(X):5d}  X={X.shape}")

In [ ]:
# HUCRE 8: Final Dogrulama + Indirme Ozeti
# Her split icin X/y shape, NaN/Inf ve sinif dagilimini kontrol eder.

print("=" * 55)
for split in ['train', 'val', 'test']:
    X = np.load(os.path.join(OUTPUT, 'sequences', split, f'X_{split}.npy'))
    y = np.load(os.path.join(OUTPUT, 'sequences', split, f'y_{split}.npy'))
    print(f"\n{split.upper()}")
    print(f"  X={X.shape}  y={y.shape}")
    print(f"  V={int((y==1).sum())}  NV={int((y==0).sum())}")
    print(f"  NaN={np.isnan(X).any()}  Inf={np.isinf(X).any()}")
    print(f"  min={X.min():.3f}  max={X.max():.3f}")

if process_log:
    pd.DataFrame(process_log).to_csv(os.path.join(OUTPUT, 'preprocessing_log.csv'), index=False)
if skipped_videos:
    pd.DataFrame(skipped_videos).to_csv(os.path.join(OUTPUT, 'skipped_videos.csv'), index=False)

print("\n" + "=" * 55)
print("Indirilecek dosyalar (Output sekmesi):")
for split in ['train', 'val', 'test']:
    for fname in [f'X_{split}.npy', f'y_{split}.npy']:
        p = os.path.join(OUTPUT, 'sequences', split, fname)
        mb = os.path.getsize(p) / 1024**2
        print(f"  sequences/{split}/{fname}  ({mb:.1f} MB)")
print("\nPreprocessing tamamlandi!")